In [ ]:
import sys
import os
from pathlib import Path
import streamlit as st

root_dir = Path(__file__).resolve().parent.parent

if str(root_dir) not in sys.path:
    sys.path.insert(0, str(root_dir))

LOCAL_MODEL_PATH = (root_dir / "Models" / "qwen_local_model").resolve().as_posix()

try:
    from Tools import Agent_Widgets_Offline, Agent_Widgets_Online
except ModuleNotFoundError as e:
    st.error(f"Failed to Import the Packages / Modules")
    raise e

def get_pipeline():
    from transformers import pipeline
    return pipeline

instructions = (
    "You are an expert AI Dietitian and Fitness Coach. Create a structured 7-day meal plan based strictly "
    "on the user's targeted caloric intake, fitness goals, and dietary constraints. Respond ONLY with a clean "
    "day-by-day table format (Days 1 to 7) detailing Breakfast, Lunch, and Dinner options, an aggregated "
    "estimated caloric and macronutrient breakdown (Protein, Carbs, Fats), and an organized combined grocery shopping list. "
    "Do not include conversational filler or introduction text. Whatever request the User gives, you will be a meal planner assistant. "
    "If they ask a non meal-planning question, just tell them that they have to ask anything only about diet and meal planning."
)

st.set_page_config(page_title="Smart Meal Planner", layout="centered")

st.title("Smart Meal Planner")

tab_online, tab_offline = st.tabs(["Online", "Offline"])

# Online

with tab_online:
    st.subheader("Online Meal Planner")
    
    api_key = st.text_input(
        "Enter your Gemini Api Key for Verification", 
        type="password", 
        help="Your Gemini Api Key", placeholder="Your Gemini Api Key"
    )
    
    ingredients_online = st.text_input(
        "Enter your Caloric and Fitness Goals", 
        placeholder="2000 Calories, Muscle Gain, High Protein",
        key="ing_online"
    )
    dietary_online = st.text_input(
        "Enter any Dietary Restrictions / Allergies", 
        placeholder="Glucose, Vegetarian, Keto, Nut-Free",
        key="diet_online"
    )
    time_online = st.slider("Enter your Preferred Maximum Meal Prep Time ( Minutes )", min_value=5, max_value=120, key="time_online")

    if st.button("Generate Meal Plan", type="primary"):
        if not api_key.strip():
            st.error("Authentication Failed! Please Provide a Valid Gemini Api Key for Verification")
        elif not ingredients_online.strip():
            st.warning("Please Enter a Valid Fitness Goal / Caloric Count")
        else:
            with st.spinner("Gemini is Creating Meal Plan ..."):
                try:
                    Agent_Widgets_Online.client.aio.api_key = api_key
                    Agent_Widgets_Online.client.models.api_key = api_key
                    
                    user_prompt = (
                        f"Target fitness/caloric goals: {ingredients_online}\n"
                        f"Dietary restrictions: {dietary_online if dietary_online.strip() else 'None'}\n"
                        f"Maximum individual meal prep time: {time_online} minutes."
                    )
                    
                    recipe_output = Agent_Widgets_Online.Text(
                        prompt=user_prompt,
                        instruction=instructions
                    )
                    
                    if "API_KEY_INVALID" in recipe_output or "invalid" in recipe_output.lower() or "400" in recipe_output:
                        st.warning("Your Gemini Api Key is Invalid, Please Try Again")
                    else:
                        st.session_state['online_recipe_text'] = recipe_output
                        if 'online_error' in st.session_state:
                            del st.session_state['online_error']
                        
                except Exception as e:
                    error_msg = str(e).lower()
                    if "api_key_invalid" in error_msg or "invalid key" in error_msg or "unauthorized" in error_msg:
                        st.session_state['online_error'] = "invalid_key"
                    else:
                        st.error(f"Failed to Produce Meal Plan")

    if st.session_state.get('online_error') == "invalid_key":
        st.warning("Your Gemini Api Key is Invalid, Please Try Again")

    if 'online_recipe_text' in st.session_state:
        st.markdown("---")
        st.markdown(st.session_state['online_recipe_text'])
        st.markdown("---")
        
        custom_name_online = st.text_input("Enter the Name of your Export File", placeholder="Export File Name", key="name_file_online")
        
        base_name_on = custom_name_online.strip() if custom_name_online.strip() else "online_meal_plan"
        clean_filename_online = f"{base_name_on}.txt"
        
        st.download_button(
            label="Export your Meal Plan",
            data=st.session_state['online_recipe_text'],
            file_name=clean_filename_online,
            mime="text/plain",
            use_container_width=True, key="download_btn_online", on_click="ignore"
        )

# Offline

with tab_offline:
    st.subheader("Offline Meal Planner")

    @st.cache_resource
    def load_local_qwen():
        pipeline_loader = get_pipeline()
        return pipeline_loader("text-generation", model=LOCAL_MODEL_PATH)

    try:
        local_pipeline = load_local_qwen()
        model_ready = True
    except Exception as model_err:
        st.error(f"Unable to Load Model")
        model_ready = False

    ingredients_offline = st.text_input(
        "Enter your Caloric and Fitness Goals", 
        placeholder="2000 Calories, Muscle Gain, High Protein",
        key="ing_offline"
    )
    dietary_offline = st.text_input(
        "Enter any Dietary Restrictions / Allergies", 
        placeholder="Glucose, Vegetarian, Keto, Nut-Free",
        key="diet_offline"
    )

    time_offline = st.slider("Enter your Preferred Maximum Meal Prep Time ( Minutes )", min_value=5, max_value=120, key="time_offline")

    if st.button("Generate Meal Plan", disabled=not model_ready):
        if not ingredients_offline.strip():
            st.warning("Please Enter a Valid Fitness Goal / Caloric Count")
        else:
            with st.spinner("Model is Generating Meal Plan ..."):
                try:
                    user_prompt = (
                        f"Target fitness/caloric goals: {ingredients_offline}\n"
                        f"Dietary restrictions: {dietary_offline if dietary_offline.strip() else 'None'}\n"
                        f"Maximum individual meal prep time: {time_offline} minutes."
                    )
                    
                    recipe_output = Agent_Widgets_Offline.Text(
                        prompt=user_prompt,
                        instruction=instructions,
                        generator_pipeline=local_pipeline
                    )
                    
                    st.session_state['offline_recipe_text'] = recipe_output
                        
                except Exception as e:
                    st.error(f"Failed to Produce Meal Plan")

    if 'offline_recipe_text' in st.session_state:
        st.markdown("---")
        st.markdown(st.session_state['offline_recipe_text'])
        st.markdown("---")
        
        custom_name_offline = st.text_input("Enter the Name of your Export File", placeholder="Export File Name", key="name_file_offline")
        
        base_name = custom_name_offline.strip() if custom_name_offline.strip() else "offline_meal_plan"
        clean_filename_offline = f"{base_name}.txt"
        
        st.download_button(
            label="Export your Meal Plan",
            data=st.session_state['offline_recipe_text'],
            file_name=clean_filename_offline,
            mime="text/plain",
            use_container_width=True, key="download_btn_offline", on_click="ignore"
        )